In [ ]:
print('Hello World!')

# 最初のセルの実行は時間がかかる

In [ ]:
%pip install hscmap

## 銀河探しゲームの領域表示

In [ ]:
from hscmap import Window
from hscmap import Vec3, SkyCoord, Angle
from hscmap.shape import Grid, Line

In [ ]:
w = Window()

In [ ]:
pdr3_dud_layer = w.dataset.tile_layers['PDR3 DUD']

In [ ]:
pdr3_dud_layer.visible = False # PDR3 DUDのデータを見えなくする

In [ ]:
center = (35.749359446327695, -5.13871964659809)
w.camera.jump_to(*center, fov=2)

## グリッド表示

In [ ]:
def draw_grid(center):
    shape = Grid(
        center=SkyCoord.from_degree(*center), 
        color=[0, 0.75, 1, 1],
        width=Angle.from_degree(1.36),
        height=Angle.from_degree(1.38),
        div_x=6,
        div_y=4,
        up=Vec3(0, 0, 1),
    )
    grid = w.regions.new_shape(name='Grid', shapes=[shape])

    def cleanup():
        grid.delete()

    return cleanup

clear = draw_grid(center)

気になる天体を右クリック → SIMBAD検索で調べる

In [ ]:
clear() # 消す

## ドラッグできる枠

In [ ]:
circle = w.regions.new_circle(center=w.camera.center, radius=0.75, color=[1, 1, 0, 1])
w.camera.fov = 2

In [ ]:
cleanup = None

def watch_on():
    return circle.center

def on_change():
    global cleanup
    if cleanup is not None:
        cleanup()
    cleanup = draw_grid(circle.center)
    circle.surface()

on_change()
watcher = w.watchers.new(watch_on=watch_on, on_change=on_change)

円の中心をドラッグすると、枠もついてくる

In [ ]:
watcher.unwatch() # ついてこなくする

## パネル番号付き枠

In [ ]:
def draw_grid_with_panel_number(center, panel_numbers=True):
    # グリッド分割数
    div_h = 6
    div_v = 4    
    # グリッド大きさ
    width = Angle.from_degree(1.36).radian
    height = Angle.from_degree(1.38).radian    
    # 焦点面軸を計算
    o = SkyCoord.from_degree(*center).as_vec3()
    ez = Vec3(0, 0, 1) 
    e1 = o.cross(ez).normalize()
    e2 = o.cross(e1).normalize()
    shapes = []
    color = [0, 1, 1, 0.75]
    for i in range(div_v + 1):
        v = (i - div_v/2) / div_v
        h1 = -0.5
        h2 = +0.5
        p1 = o + (width * h1 * e1) + (height * v * e2)
        p2 = o + (width * h2 * e1) + (height * v * e2)
        shapes.append(Line(p1, p2, color))
    for j in range(div_h + 1):
        h = (j - div_h/2) / div_h
        v1 = -0.5
        v2 = +0.5
        p1 = o + (width * h * e1) + (height * v1 * e2)
        p2 = o + (width * h * e1) + (height * v2 * e2)
        shapes.append(Line(p1, p2, color))
    grid = w.regions.new_shape(shapes=shapes, name='Grid')
    # パネル番号表示
    texts = []
    if panel_numbers:
        for i in range(div_v):
            v = (i - div_v/2) / div_v    
            for j in range(div_h):
                h = (j - div_h/2) / div_h
                p = o + (width * h * e1) + (height * v * e2)
                coord = SkyCoord.from_vec3(p)
                t = w.regions.new_text(
                    text=f'Panel Number/{len(texts)}',
                    position=(coord.ra.degree, coord.dec.degree),
                    color=[1, 1, 1, 1],
                )
                texts.append(t)
    
    def cleanup():
        grid.delete()
        for t in texts:
            t.delete()

    return cleanup

clear = draw_grid_with_panel_number(w.camera.center)

In [ ]:
clear()